# Fine-Tuning Qwen2.5-0.5B for Ecosystem Intent & Skill Routing

This notebook fine-tunes **`Qwen2.5-0.5B-Instruct`** using **Unsloth** and LoRA to serve as the ultra-fast local router for `adaptive-workflow` and `Fleet-Orchestrator`.

### Runtime Setting:
- Select **Runtime > Change runtime type > T4 GPU** (Free tier).
- Total training time on T4: **~4 to 6 minutes**.
- Output: Quantized **`q4_k_m` GGUF** model file (~380 MB) downloaded directly to your machine.

In [ ]:
%%capture
!pip install unsloth "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

## 1. Load Model & Tokenizer in 4-bit

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
dtype = None  # None for auto-detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 42,
)

## 2. Ingest Dataset
Upload `intent_routing_train.jsonl` and `intent_routing_eval.jsonl` via the Colab files sidebar, or run the cell below to load them.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("json", data_files={
    "train": "intent_routing_train.jsonl",
    "eval": "intent_routing_eval.jsonl"
})

def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input_text, output in zip(instructions, inputs, outputs):
        text = f"<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{input_text}<|im_end|>\n<|im_start|>assistant\n{output}<|im_end|>"
        texts.append(text)
    return {"text": texts}

dataset = dataset.map(formatting_prompts_func, batched = True)
print(f"Loaded {len(dataset['train'])} train samples, {len(dataset['eval'])} eval samples")

## 3. Train LoRA Adapters

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset["train"],
    eval_dataset = dataset["eval"],
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 4,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 42,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 4. Test Structured Inference Live

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Fix the regression in test_warehouse_mem_sim.py where queue latency was calculating as zero.",
    "Scaffold IOE BE semester IV-II notes for CE 752 and EX 756 with syllabus markdown hubs.",
    "Synthesize a 4th-order Chebyshev low-pass IIR filter for 10kHz audio sampling via Bilinear Transform.",
    "Lock the private research directory using kernel NTFS ACL security in Win-Vault."
]

for prompt in test_prompts:
    prompt_text = f"<|im_start|>system\nYou are the high-speed Intent and Skill Router for the Aaradhya development ecosystem. Classify the incoming user intent into the exact lifecycle archetype, tier, 2D matrix cell, primary skill, supporting skills, and velocity profile in strict JSON format.<|im_end|>\n<|im_start|>user\n{prompt}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt_text], return_tensors = "pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens = 150, use_cache = True)
    res = tokenizer.batch_decode(outputs)[0].split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()
    print(f"\nPrompt: {prompt}")
    print(f"Output:\n{res}")

## 5. Export to GGUF (Q4_K_M) & Download to Laptop

In [ ]:
import glob
from google.colab import files

# Export quantized GGUF directly (~380 MB)
model.save_pretrained_gguf("qwen_intent_router_q4", tokenizer, quantization_method = "q4_k_m")

gguf_files = glob.glob("qwen_intent_router_q4/*.gguf")
if gguf_files:
    print(f"Found GGUF: {gguf_files[0]}. Triggering download...")
    files.download(gguf_files[0])
else:
    print("Error: GGUF file not found in qwen_intent_router_q4/")